# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_04 — AI_SUMMARIZE_PANDAS

## 1. Objetivos y requisitos
Generar resúmenes por fila, comparar una columna con múltiples columnas, validar fidelidad y auditar tokens y CU. Requiere runtime Python de Fabric compatible, AI Functions habilitadas y SynapseML para pandas. Utilizar únicamente datos autorizados; las inferencias consumen capacidad.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo gpt-5-mini, concurrencia de laboratorio 10 y barra de progreso con estadísticas. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.summarize con pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/summarize)

```python
df["summaries"] = df["text"].ai.summarize()
df["summaries"] = df.ai.summarize()
```

| Parámetro | Tipo | Uso |
|---|---|---|
| `instructions` | str, opcional | Orienta longitud, tono, audiencia y contenido; si se omite se usa el comportamiento estándar. |
| `column_type` | str, opcional | Para una Series de rutas, utilizar `path` según la documentación multimodal. |

Devuelve una Series de resúmenes por fila. Para texto nulo en una columna, devuelve nulo. Una llamada sobre DataFrame considera todas las columnas seleccionadas, no produce un resumen global. Los ejemplos siguientes son propios del laboratorio.

## 4. Dataset ficticio
Mismos seis registros del NB_04 PySpark V2; cinco descripciones y una nula.

In [3]:
data = [
    (1, "Incidente", "Alta", "Resuelto", "Durante la carga nocturna, el pipeline de ventas falló por una conexión expirada. El equipo renovó las credenciales, reejecutó el proceso y verificó que los registros llegaron completos al destino."),
    (2, "Rendimiento", "Media", "En análisis", "El reporte comercial demora más de lo habitual. Se detectaron consultas con filtros poco selectivos. El equipo revisará el modelo semántico y realizará pruebas antes de aplicar cambios."),
    (3, "Solicitud", "Baja", "Pendiente", "El área financiera solicita un tablero que consolide ingresos y gastos mensuales. Se requiere definir los indicadores y validar las fuentes con los responsables del negocio."),
    (4, "Calidad", "Alta", "Mitigado", "Se identificaron registros duplicados en una tabla de pedidos. Se aisló la carga afectada y se aplicó una deduplicación temporal. Falta corregir la lógica de origen."),
    (5, "Disponibilidad", "Media", "Resuelto", "Una actualización del servicio dejó temporalmente indisponible el acceso a un conjunto de datos. El servicio se restauró y las pruebas confirmaron el acceso normal."),
    (6, "Soporte", "Baja", "Pendiente", None)
]
df = pd.DataFrame(data, columns=["id", "tipo", "prioridad", "estado", "descripcion"])
display(df)

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7abb0120-5f8d-4664-9ad0-00bf3f93a704)

## 5. Resumen de una columna
Conservar la Series original y sus estadísticas antes de asignarla al DataFrame. El tiempo mide la llamada, no el tiempo total del notebook.

In [4]:
runs = []
def ejecutar(nombre, entrada, **kwargs):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    input_bytes = int(entrada.memory_usage(deep=True).sum() if isinstance(entrada, pd.DataFrame) else entrada.memory_usage(deep=True))
    t0 = perf_counter()
    resultado = entrada.ai.summarize(**kwargs)
    elapsed = perf_counter() - t0
    raw = resultado.ai.stats
    runs.append((nombre, run_id, started, elapsed, raw, resultado, len(entrada), input_bytes))
    return resultado

resumen = ejecutar("default", df["descripcion"])
display(df.assign(resumen=resumen))

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 6, Finished, Available, Finished, False)

ai.summarize: 100%|██████████| 6/6 [00:03<00:00,  1.53it/s, cached=0, in=729, out=632, CU-s=43.10]                             


SynapseWidget(Synapse.DataFrame, 32a5f288-02da-40f1-aa9e-1d6ec34741e5)

## 6. Resumen ejecutivo de múltiples columnas
Excluir la descripción nula como en PySpark V2. La segunda llamada incorpora contexto adicional e instrucciones, por lo que no mide aisladamente el efecto de las instrucciones.

In [5]:
df_ejecutivo = df.loc[df["descripcion"].notna(), ["id", "tipo", "prioridad", "estado", "descripcion"]].copy()
resumen_ejecutivo = ejecutar("custom", df_ejecutivo, instructions="Redacta en español un resumen ejecutivo de máximo 40 palabras. Incluye prioridad, estado, impacto y acciones pendientes si constan en los datos. No incluyas el identificador ni inventes información.")
display(df_ejecutivo.assign(resumen_ejecutivo=resumen_ejecutivo))

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 7, Finished, Available, Finished, False)

ai.summarize: 100%|██████████| 5/5 [00:08<00:00,  1.77s/it, cached=0, in=1.31k, out=1.99k, CU-s=134.62]                            


SynapseWidget(Synapse.DataFrame, eedaca87-ffdf-4b54-9896-43d98f0dda09)

## 7. Validación y comparación
Medir reducción de caracteres (no tokens), revisar nulos y comprobar manualmente hechos, estado, impacto y acciones. El límite de palabras es una instrucción, no una garantía.

In [7]:
resultado = df.assign(resumen=resumen, resumen_ejecutivo=resumen_ejecutivo)

# Asegurar que las columnas de texto se traten como cadenas antes de usar .str
resultado["descripcion"] = resultado["descripcion"].astype("string")
resultado["resumen"] = resultado["resumen"].astype("string")
resultado["resumen_ejecutivo"] = resultado["resumen_ejecutivo"].astype("string")

# Longitud en caracteres del texto original y del resumen
resultado["caracteres_original"] = resultado.descripcion.str.len()
resultado["caracteres_resumen"] = resultado.resumen.str.len()

# Evitar división por cero: solo considerar textos con longitud > 0
base = resultado.caracteres_original.where(resultado.caracteres_original.gt(0))
resultado["reduccion_pct"] = 100 * (1 - resultado.caracteres_resumen / base)

# Métricas del resumen ejecutivo (número de palabras y si supera el límite solicitado)
resultado["palabras_ejecutivo"] = resultado.resumen_ejecutivo.str.split().str.len()
resultado["supera_limite"] = resultado.palabras_ejecutivo.astype("Int64").gt(40)

display(resultado)
print("Resúmenes nulos con descripción presente:", int((resultado.descripcion.notna() & resultado.resumen.isna()).sum()))

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b66b4c86-0bc2-4dfa-9568-29a543992d74)

Resúmenes nulos con descripción presente: 0


## 8. Estadísticas y tokens
Consultar cada ejecución por separado. No reemplazar métricas ausentes por cero.

In [8]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")
for nombre, _, _, _, raw, _, _, _ in runs:
    print(nombre)
    display(raw)

StatementMeta(, 75a691df-a5ab-4bb7-9a9c-86406a1ee15a, 10, Finished, Available, Finished, True)

default


SynapseWidget(Synapse.DataFrame, 3866ccb5-a642-4e0d-8350-3c364e13177e)

custom


SynapseWidget(Synapse.DataFrame, 3752c1e8-95f1-4e59-a993-05eeafc867ae)

## 9. Estimación de CU-segundos
Tarifas de referencia de la serie para gpt-5-mini integrado, en CU-s por 1.000 tokens:

| Entrada sin caché | Entrada en caché | Salida |
|---:|---:|---:|
| 8,40 | 0,84 | 67,23 |

$CU_s=((I-C)\times8.40+C\times0.84+O\times67.23)/1000$

C es un subconjunto de I. Los tokens de razonamiento no se suman nuevamente a O. Ejemplo hipotético: I=1000, C=200, O=100 da 13,611 CU-s. No representa una ejecución real ni el consumo total del runtime. Verificar vigencia en [facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Si faltan métricas o el modelo/cliente no coincide, no estimar.

## 10. Auditoría individual y acumulada
Registrar ambas llamadas, tiempo y memoria aproximada de DataFrames (no memoria máxima del proceso).

In [ ]:
records = []
for nombre, run_id, started, elapsed, raw, resultado_run, num_rows, input_bytes in runs:
    stats = normalizar(raw)
    def metric(k):
        v = stats.get(k)
        return None if v is None or pd.isna(v) else int(v)
    I,C,O = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens")]
    cu = None
    if (stats.get("model") in (MODEL_NAME,"gpt-5-mini-2025-08-07") and stats.get("client_type")=="fabric_llm_endpoint" and all(v is not None for v in (I,C,O)) and 0<=C<=I and O>=0):
        cu = ((I-C)*8.40+C*0.84+O*67.23)/1000
    audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_04_AI_SUMMARIZE_PANDAS", operation="ai.summarize:"+nombre, model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=num_rows, client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=input_bytes, output_bytes=int(resultado_run.memory_usage(deep=True)), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key]=metric(key)
    audit["execution_status"]="SUCCESS" if audit["num_successful"]==num_rows else "INCOMPLETE"
    records.append(audit)
audit_df=pd.DataFrame(records)
display(audit_df)
print("CU-s acumulados:", audit_df.estimated_cu_seconds.sum(min_count=len(audit_df)))

## 11. Persistencia opcional en Delta
Requiere Lakehouse adjunto, deltalake y pyarrow. Persiste en Files, no registra una tabla SQL. Deshabilitada por defecto.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 12. Multimodalidad (opcional)
Usar archivos autorizados y revisar [requisitos multimodales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). Consumo independiente, fuera de la auditoría principal.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = pd.Series(["/lakehouse/default/Files/ejemplo.pdf"])
    resultado_archivos = archivos.ai.summarize(instructions="Resume el archivo en español.", column_type="path")
    display(resultado_archivos)
    display(resultado_archivos.ai.stats)

## 13. Comparación y conclusiones
Pandas opera sobre Series con ejecución inmediata; PySpark devuelve un DataFrame distribuido con evaluación diferida. La concurrencia de solicitudes no equivale al paralelismo Spark. Las métricas de memoria representan objetos, no el pico del proceso.

Referencias: [método oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/summarize), [configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration), [facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).